In [1]:
import pandas as pd

df = pd.read_csv(
    "../data/processed/restaurant_inspections.csv.gz",
    parse_dates=["INSP_DATE"],
    low_memory=False
)

print(df.shape)

(681641, 144)


In [2]:
df["LICENSE_NO"].nunique()

86492

In [3]:
(
    df.groupby(
        ["LICENSE_NO", "DBA_NAME", "LOC_ADDRESS"]
    )
    .size()
    .sort_values(ascending=False)
    .head(25)
)

LICENSE_NO  DBA_NAME                        LOC_ADDRESS                                  
1102633     MOM'S OG                        1017 W UNIVERSITY AVE                            65
6217228     LITTLE INDIA                    25000 US HWY 19 N                                51
7405560     CHINA MASTER                    1700 W INTERNATIONAL SPEEDWAY BLVD, SUITE 148    48
6217167     SANTA FE MEXICAN GRILL          800 CLEARWATER LARGO RD                          43
2613020     CASA DORA ITALIAN CAFE          108 E FORSYTH ST                                 42
6011515     BLUE ANCHOR                     804 E ATLANTIC AVE                               41
1101752     MR HAN RESTAURANT               6944 NW 10 PL                                    40
7407198     THE PARCHED OAK                 145 N WOODLAND BLVD                              39
2613945     LA NOPALERA MEXICAN RESTAURANT  8206 PHILIPS HWY STE 29                          38
2334107     IRON SUSHI                      90

In [4]:
(
    df.groupby(
        ["LICENSE_NO", "DBA_NAME", "LOC_ADDRESS"]
    )["HIGH_VIOL"]
    .sum()
    .sort_values(ascending=False)
    .head(25)
)

LICENSE_NO  DBA_NAME                             LOC_ADDRESS                                  
6213109     NAME VIET BISTRO MACHI               8150 N 49 ST                                     161.0
7403361     THE GRILLE AT RIVERVIEW              101 FLAGLER AVE                                  142.0
2613020     CASA DORA ITALIAN CAFE               108 E FORSYTH ST                                 122.0
6217167     SANTA FE MEXICAN GRILL               800 CLEARWATER LARGO RD                          122.0
6217228     LITTLE INDIA                         25000 US HWY 19 N                                114.0
6019913     LA GRANJA RESTAURANT                 10590 W FOREST HILL BLVD                         112.0
6021961     LEMONGRASS ASIAN BISTRO              10312 FOREST HILL BLVD STE 204                   104.0
2612750     CASA MARINA HOTEL & RESTAURANT       691 N 1 ST                                       103.0
2652482     MIS DELICIAS LATINAS                 4241 UNIVERSITY BLVD S  

In [5]:
summary = (
    df.groupby(
        ["LICENSE_NO", "DBA_NAME", "LOC_ADDRESS"]
    )
    .agg(
        inspections=("INSP_NO", "count"),
        high_priority=("HIGH_VIOL", "sum")
    )
)

summary["per_inspection"] = (
    summary["high_priority"]
    / summary["inspections"]
)

summary[
    summary["inspections"] >= 5
].sort_values(
    "per_inspection",
    ascending=False
).head(25)

,,,inspections,high_priority,per_inspection
LICENSE_NO,DBA_NAME,LOC_ADDRESS,,,
5911115,MAIN GATE FLEA MARKET FOOD COURT #1,5407 W IRLO BRONSON MEMORIAL HWY,10,72.0,7.200000
2652281,MARIA GUADALUPE GUTIERREZ,10014 ATLANTIC BLVD,14,96.0,6.857143
6012682,SUSHI YAMI,6177 JOG RD,5,34.0,6.800000
5815344,HABIBI LEBANESE GRILL,8001 S ORANGE BLOSSOM RD STE 1510,5,33.0,6.600000
6213109,NAME VIET BISTRO MACHI,8150 N 49 ST,26,161.0,6.192308
7406995,DON PEPPER'S MEXICAN GRILL & CANTINA,794 S ATLANTIC AVE,7,42.0,6.000000
6009906,POPPIES RESTAURANT & DELI,4900 LINTON BLVD,8,46.0,5.750000
2614216,WILD WING CAFE,4555 SOUTHSIDE BLVD,5,28.0,5.600000
6019135,SUSHI YAMA,10260 FOREST HILL BLVD,9,49.0,5.444444


In [6]:
restaurant_summary = (
    df.groupby(
        ["LICENSE_NO", "DBA_NAME", "LOC_ADDRESS", "LOC_CITY"]
    )
    .agg(
        inspections=("INSP_NO", "count"),
        high_priority=("HIGH_VIOL", "sum"),
        total_violations=("VIOLATIONS", "sum")
    )
    .reset_index()
)

restaurant_summary["high_priority_per_inspection"] = (
    restaurant_summary["high_priority"]
    / restaurant_summary["inspections"]
)

restaurant_summary["total_violations_per_inspection"] = (
    restaurant_summary["total_violations"]
    / restaurant_summary["inspections"]
)

restaurant_summary = restaurant_summary[
    restaurant_summary["inspections"] >= 10
]

restaurant_summary.sort_values(
    "high_priority_per_inspection",
    ascending=False
).head(30)

,LICENSE_NO,DBA_NAME,LOC_ADDRESS,LOC_CITY,inspections,high_priority,total_violations,high_priority_per_inspection,total_violations_per_inspection
80600,5911115,MAIN GATE FLEA MARKET FOOD COURT #1,5407 W IRLO BRONSON MEMORIAL HWY,KISSIMMEE,10,72.0,227,7.200000,22.700000
40070,2652281,MARIA GUADALUPE GUTIERREZ,10014 ATLANTIC BLVD,JACKSONVILLE,14,96.0,104,6.857143,7.428571
91740,6213109,NAME VIET BISTRO MACHI,8150 N 49 ST,PINELLAS PARK,26,161.0,560,6.192308,21.538462
84807,6020058,PEPPERMINT THAI,11328 OKEECHOBEE BLVD,ROYAL PALM BEACH,13,69.0,164,5.307692,12.615385
83722,6013433,KING'S SUPER BUFFET,4270 OKEECHOBEE BLVD,WEST PALM BEACH,12,61.0,104,5.083333,8.666667
91739,6213109,LOON FONG RESTAURANT,8150 N 49 STREET,PINELLAS PARK,13,64.0,266,4.923077,20.461538
84395,6019285,CHINA SKY,7040 SEMINOLE PRATT WHITNEY RD,LOXAHATCHEE,12,58.0,105,4.833333,8.750000
74440,5813948,THE CRAZY COCONUT,5406 HANSEL AVE,ORLANDO,11,52.0,256,4.727273,23.272727
108698,7406365,CHINA HOUSE 168 INC,1844 RIDGEWOOD AVE,HOLLY HILL,17,79.0,306,4.647059,18.000000
76214,5815515,YH SEAFOOD CLUBHOUSE,8081 TURKEY LAKE RD STE 700,ORLANDO,16,74.0,353,4.625000,22.062500


In [7]:
(
    df.groupby("INSPTYPE")
      .agg(
          inspections=("INSP_NO", "count"),
          avg_high_priority=("HIGH_VIOL", "mean"),
          avg_total=("VIOLATIONS", "mean")
      )
      .sort_values("avg_high_priority", ascending=False)
)

,inspections,avg_high_priority,avg_total
INSPTYPE,,,
Complaint Full,55960,1.305642,6.352823
Routine - Food,567371,1.010604,5.210765
Complaint Partial,7398,0.418120,1.637470
Food-Licensing Inspection,50859,0.264286,1.713168
Routine - Lodging,25,0.080000,0.560000
Disaster Response,28,0.000000,0.392857


In [8]:
most_inspected = (
    df.groupby(
        ["LICENSE_NO", "DBA_NAME", "LOC_ADDRESS"]
    )
    .size()
    .sort_values(ascending=False)
)

most_inspected.head(10)

LICENSE_NO  DBA_NAME                        LOC_ADDRESS                                  
1102633     MOM'S OG                        1017 W UNIVERSITY AVE                            65
6217228     LITTLE INDIA                    25000 US HWY 19 N                                51
7405560     CHINA MASTER                    1700 W INTERNATIONAL SPEEDWAY BLVD, SUITE 148    48
6217167     SANTA FE MEXICAN GRILL          800 CLEARWATER LARGO RD                          43
2613020     CASA DORA ITALIAN CAFE          108 E FORSYTH ST                                 42
6011515     BLUE ANCHOR                     804 E ATLANTIC AVE                               41
1101752     MR HAN RESTAURANT               6944 NW 10 PL                                    40
7407198     THE PARCHED OAK                 145 N WOODLAND BLVD                              39
2613945     LA NOPALERA MEXICAN RESTAURANT  8206 PHILIPS HWY STE 29                          38
2334107     IRON SUSHI                      90

In [9]:
restaurant = df[df["LICENSE_NO"] == 1102633]

restaurant = restaurant[
    [
        "INSP_DATE",
        "INSPTYPE",
        "DISPOSITION",
        "HIGH_VIOL",
        "VIOLATIONS"
    ]
].sort_values("INSP_DATE")

restaurant

,INSP_DATE,INSPTYPE,DISPOSITION,HIGH_VIOL,VIOLATIONS
59522,2021-07-23,Routine - Food,Administrative complaint recommended,5.0,23
58788,2021-07-26,Routine - Food,Call Back - Complied,0.0,0
40684,2021-09-09,Complaint Full,Administrative complaint recommended,7.0,26
39985,2021-09-10,Complaint Full,"Call Back - Extension given, pending",0.0,1
14113,2021-11-16,Complaint Full,Call Back - Admin. complaint recommended,0.0,1
...,...,...,...,...,...
619572,2026-06-17,Complaint Full,Call Back - Admin. complaint recommended,4.0,6
618128,2026-06-23,Complaint Full,Call Back - Admin. complaint recommended,1.0,3
618256,2026-06-23,Routine - Food,Emergency order recommended,1.0,1
617812,2026-06-24,Routine - Food,Emergency Order Callback Not Complied,1.0,1


In [10]:
(
    df.groupby("INSPTYPE")
      .agg(
          inspections=("INSP_NO", "count"),
          failed=("DISPOSITION", lambda x: x.str.contains("Admin|Complaint", case=False, na=False).sum())
      )
      .assign(failure_rate=lambda x: x["failed"] / x["inspections"])
      .sort_values("failure_rate", ascending=False)
)

,inspections,failed,failure_rate
INSPTYPE,,,
Complaint Full,55960,6778,0.121122
Routine - Food,567371,48292,0.085115
Complaint Partial,7398,558,0.075426
Food-Licensing Inspection,50859,406,0.007983
Disaster Response,28,0,0.000000
Routine - Lodging,25,0,0.000000


In [11]:
(
    df[df["INSPTYPE"].str.contains("Complaint", na=False)]
    ["DISPOSITION"]
    .value_counts()
)

DISPOSITION
Inspection Completed - No Further Action    29058
Call Back - Complied                        10978
Warning Issued                               9062
Administrative complaint recommended         5022
Call Back - Admin. complaint recommended     2289
Emergency order recommended                  2044
Call Back - Extension given, pending         2008
Emergency Order Callback Complied            1527
Emergency Order Callback Not Complied         960
Emergency Order Callback Time Extension       311
Assigned to Inspector                          54
Administrative determination recommended       14
Not available electronically                   12
Admin. Complaint Callback Complied              8
Insp. Completed - Warning Given, Pending        5
Admin. Complaint Callback Not Complied          3
Allegation Not Observed                         3
Name: count, dtype: int64

In [12]:
(
    df.groupby("CNTY_DESC")
      .agg(
          inspections=("INSP_NO", "count"),
          avg_high=("HIGH_VIOL", "mean")
      )
      .query("inspections >= 500")
      .sort_values("avg_high", ascending=False)
      .head(20)
)

,inspections,avg_high
CNTY_DESC,,
Baker,560,1.514286
Levy,1299,1.434950
Volusia,19698,1.384240
Martin,6300,1.376825
Palm Beach,46906,1.340511
Duval,35353,1.309681
Putnam,1952,1.296258
Clay,4658,1.295620
Bradford,617,1.277147


In [13]:
inspection_type_counts = (
    df["INSPTYPE"]
    .value_counts(dropna=False)
    .rename_axis("inspection_type")
    .reset_index(name="inspections")
)

inspection_type_counts["percent"] = (
    inspection_type_counts["inspections"]
    / inspection_type_counts["inspections"].sum()
    * 100
)

inspection_type_counts

,inspection_type,inspections,percent
0,Routine - Food,567371,83.236044
1,Complaint Full,55960,8.209600
2,Food-Licensing Inspection,50859,7.461259
3,Complaint Partial,7398,1.085322
4,Disaster Response,28,0.004108
5,Routine - Lodging,25,0.003668


In [14]:
complaints = df[
    df["INSPTYPE"].isin(
        ["Complaint Full", "Complaint Partial"]
    )
].copy()

print("Complaint inspections:", len(complaints))

Complaint inspections: 63358


In [15]:
complaint_findings = pd.Series({
    "Zero high-priority violations":
        (complaints["HIGH_VIOL"] == 0).mean() * 100,

    "At least one high-priority violation":
        (complaints["HIGH_VIOL"] >= 1).mean() * 100,

    "Three or more high-priority violations":
        (complaints["HIGH_VIOL"] >= 3).mean() * 100
})

complaint_findings.round(1)

Zero high-priority violations             47.8
At least one high-priority violation      52.1
Three or more high-priority violations    17.0
dtype: float64

In [16]:
complaints["DISPOSITION"].value_counts(dropna=False)

DISPOSITION
Inspection Completed - No Further Action    29058
Call Back - Complied                        10978
Warning Issued                               9062
Administrative complaint recommended         5022
Call Back - Admin. complaint recommended     2289
Emergency order recommended                  2044
Call Back - Extension given, pending         2008
Emergency Order Callback Complied            1527
Emergency Order Callback Not Complied         960
Emergency Order Callback Time Extension       311
Assigned to Inspector                          54
Administrative determination recommended       14
Not available electronically                   12
Admin. Complaint Callback Complied              8
Insp. Completed - Warning Given, Pending        5
Admin. Complaint Callback Not Complied          3
Allegation Not Observed                         3
Name: count, dtype: int64

In [17]:
complaint_outcomes = (
    complaints["DISPOSITION"]
    .value_counts(dropna=False)
    .rename_axis("outcome")
    .reset_index(name="inspections")
)

complaint_outcomes["percent"] = (
    complaint_outcomes["inspections"]
    / complaint_outcomes["inspections"].sum()
    * 100
)

complaint_outcomes

,outcome,inspections,percent
0,Inspection Completed - No Further Action,29058,45.863190
1,Call Back - Complied,10978,17.326936
2,Warning Issued,9062,14.302850
3,Administrative complaint recommended,5022,7.926387
4,Call Back - Admin. complaint recommended,2289,3.612803
5,Emergency order recommended,2044,3.226112
6,"Call Back - Extension given, pending",2008,3.169292
7,Emergency Order Callback Complied,1527,2.410114
8,Emergency Order Callback Not Complied,960,1.515199
9,Emergency Order Callback Time Extension,311,0.490861


In [18]:
complaints_by_year = (
    complaints.groupby("fiscal_year")
    .agg(
        complaint_inspections=("INSP_NO", "count"),
        avg_high_priority=("HIGH_VIOL", "mean"),
        no_high_priority=(
            "HIGH_VIOL",
            lambda x: (x == 0).mean() * 100
        ),
        three_or_more=(
            "HIGH_VIOL",
            lambda x: (x >= 3).mean() * 100
        )
    )
    .reset_index()
)

complaints_by_year.round(2)

,fiscal_year,complaint_inspections,avg_high_priority,no_high_priority,three_or_more
0,2021-22,10874,1.33,42.98,19.38
1,2022-23,12161,1.25,45.07,17.49
2,2023-24,12687,1.22,48.22,17.18
3,2024-25,13368,1.14,50.00,15.99
4,2025-26,14268,1.11,51.35,15.61


In [19]:
comparison = (
    df[df["INSPTYPE"].isin(["Routine - Food", "Complaint Full"])]
    .groupby("INSPTYPE")
    .agg(
        inspections=("INSP_NO", "count"),
        avg_high=("HIGH_VIOL", "mean"),
        zero_high=("HIGH_VIOL", lambda x: (x == 0).mean() * 100),
        three_plus=("HIGH_VIOL", lambda x: (x >= 3).mean() * 100),
        avg_total=("VIOLATIONS", "mean")
    )
    .round(2)
)

comparison

,inspections,avg_high,zero_high,three_plus,avg_total
INSPTYPE,,,,,
Complaint Full,55960,1.31,44.55,18.84,6.35
Routine - Food,567371,1.01,51.12,13.20,5.21


In [20]:
inspection_type_counts.to_csv(
    "../data/processed/chart_inspection_types.csv",
    index=False
)

In [21]:
comparison.to_csv(
    "../data/processed/chart_routine_vs_complaint.csv"
)

In [22]:
complaint_outcomes.to_csv(
    "../data/processed/chart_complaint_outcomes.csv",
    index=False
)

In [23]:
complaints_by_year.to_csv(
    "../data/processed/chart_complaints_over_time.csv",
    index=False
)

In [24]:
restaurant_counts = (
    df.groupby(["LICENSE_NO", "DBA_NAME"])
      .agg(
          inspections=("INSP_NO", "count"),
          inspection_types=("INSPTYPE", "nunique"),
          dispositions=("DISPOSITION", "nunique")
      )
      .query("inspections >= 15")
      .sort_values(
          ["inspection_types", "dispositions", "inspections"],
          ascending=False
      )
)

restaurant_counts.head(25)

,,inspections,inspection_types,dispositions
LICENSE_NO,DBA_NAME,,,
1102633,MOM'S OG,65,4,10
2613891,INDULGENCE SOUTHERN BISTRO,29,4,10
3100032,LIGHTHOUSE RESTAURANT,29,4,10
6201120,TREASURE ISLAND R BAR,22,4,10
6215207,LOS HERNANDEZ FAMILY RESTAURANT,22,4,10
6501492,SONIC DRIVE IN 4292,21,4,10
7407198,THE PARCHED OAK,39,4,9
3911556,CHINA BUFFET,32,4,9
6217123,DISCOVERY INDIAN CUISINE,32,4,9


In [25]:
restaurant = (
    df[df["LICENSE_NO"] == 7407198]
    .sort_values("INSP_DATE")
)

restaurant[
    [
        "INSP_DATE",
        "INSPTYPE",
        "DISPOSITION",
        "HIGH_VIOL",
        "VIOLATIONS"
    ]
]

,INSP_DATE,INSPTYPE,DISPOSITION,HIGH_VIOL,VIOLATIONS
64540,2021-07-13,Complaint Full,Administrative complaint recommended,6.0,10
64220,2021-07-14,Complaint Full,Call Back - Complied,0.0,1
53354,2021-08-09,Complaint Partial,Warning Issued,1.0,1
53209,2021-08-10,Routine - Food,Emergency order recommended,3.0,5
53276,2021-08-10,Complaint Partial,"Call Back - Extension given, pending",1.0,1
52237,2021-08-11,Routine - Food,Emergency Order Callback Complied,0.0,1
52415,2021-08-11,Complaint Partial,Call Back - Complied,0.0,0
16642,2021-11-09,Routine - Food,Emergency order recommended,5.0,19
15910,2021-11-10,Routine - Food,Emergency Order Callback Not Complied,2.0,11
15911,2021-11-10,Routine - Food,Emergency Order Callback Time Extension,0.0,7


In [26]:
restaurant = (
    df[df["LICENSE_NO"] == 1102633]
    .sort_values("INSP_DATE")
)

restaurant[
    [
        "INSP_DATE",
        "INSPTYPE",
        "DISPOSITION",
        "HIGH_VIOL",
        "VIOLATIONS"
    ]
]

,INSP_DATE,INSPTYPE,DISPOSITION,HIGH_VIOL,VIOLATIONS
59522,2021-07-23,Routine - Food,Administrative complaint recommended,5.0,23
58788,2021-07-26,Routine - Food,Call Back - Complied,0.0,0
40684,2021-09-09,Complaint Full,Administrative complaint recommended,7.0,26
39985,2021-09-10,Complaint Full,"Call Back - Extension given, pending",0.0,1
14113,2021-11-16,Complaint Full,Call Back - Admin. complaint recommended,0.0,1
...,...,...,...,...,...
619572,2026-06-17,Complaint Full,Call Back - Admin. complaint recommended,4.0,6
618128,2026-06-23,Complaint Full,Call Back - Admin. complaint recommended,1.0,3
618256,2026-06-23,Routine - Food,Emergency order recommended,1.0,1
617812,2026-06-24,Routine - Food,Emergency Order Callback Not Complied,1.0,1


In [27]:
chart1 = inspection_type_counts.copy()

# Combine smaller categories
major = [
    "Routine - Food",
    "Complaint Full",
    "Complaint Partial",
    "Food Licensing"
]

chart1["inspection_type"] = chart1["inspection_type"].where(
    chart1["inspection_type"].isin(major),
    "Other"
)

chart1 = (
    chart1.groupby("inspection_type", as_index=False)
          .agg(
              inspections=("inspections", "sum"),
              percent=("percent", "sum")
          )
)

chart1 = chart1.sort_values(
    "inspections",
    ascending=False
)

chart1

,inspection_type,inspections,percent
3,Routine - Food,567371,83.236044
0,Complaint Full,55960,8.209600
2,Other,50912,7.469034
1,Complaint Partial,7398,1.085322


In [28]:
chart1.to_csv(
    "../data/processed/chart1_inspection_types.csv",
    index=False
)

In [29]:
chart2 = comparison.reset_index()

chart2.to_csv(
    "../data/processed/chart2_routine_vs_complaint.csv",
    index=False
)

chart2

,INSPTYPE,inspections,avg_high,zero_high,three_plus,avg_total
0,Complaint Full,55960,1.31,44.55,18.84,6.35
1,Routine - Food,567371,1.01,51.12,13.20,5.21


In [30]:
chart2 = comparison.reset_index()

chart2 = chart2.rename(columns={
    "INSPTYPE": "inspection_type",
    "avg_high": "Average high-priority violations",
    "avg_total": "Average total violations",
    "three_plus": "Inspections with 3+ high-priority violations (%)"
})

chart2

,inspection_type,inspections,Average high-priority violations,zero_high,Inspections with 3+ high-priority violations (%),Average total violations
0,Complaint Full,55960,1.31,44.55,18.84,6.35
1,Routine - Food,567371,1.01,51.12,13.20,5.21


In [31]:
chart2.to_csv(
    "../data/processed/chart2_comparison.csv",
    index=False
)

In [32]:
comparison

,inspections,avg_high,zero_high,three_plus,avg_total
INSPTYPE,,,,,
Complaint Full,55960,1.31,44.55,18.84,6.35
Routine - Food,567371,1.01,51.12,13.20,5.21


In [33]:
chart2 = comparison.reset_index()

chart2

,INSPTYPE,inspections,avg_high,zero_high,three_plus,avg_total
0,Complaint Full,55960,1.31,44.55,18.84,6.35
1,Routine - Food,567371,1.01,51.12,13.20,5.21


In [34]:
chart2.to_csv(
    "../data/processed/chart2_comparison.csv",
    index=False
)

In [35]:
chart2a = pd.DataFrame({
    "measure": ["Average high-priority violations"],
    "Routine": [
        comparison.loc["Routine - Food", "avg_high"]
    ],
    "Complaint": [
        comparison.loc["Complaint Full", "avg_high"]
    ]
})

chart2a

,measure,Routine,Complaint
0,Average high-priority violations,1.01,1.31


In [36]:
chart2a.to_csv(
    "../data/processed/chart2a_high_priority.csv",
    index=False
)

In [37]:
chart2b = pd.DataFrame({
    "Type": ["Per inspection"],
    "Routine": [
        comparison.loc["Routine - Food", "avg_total"]
    ],
    "Complaint": [
        comparison.loc["Complaint Full", "avg_total"]
    ]
})

chart2b

,Type,Routine,Complaint
0,Per inspection,5.21,6.35


In [38]:
chart2b.to_csv(
    "../data/processed/chart2b_total_violations.csv",
    index=False
)

In [39]:
chart2c = pd.DataFrame({
    "Type": ["Share of inspections"],
    "Routine": [
        comparison.loc["Routine - Food", "three_plus"]
    ],
    "Complaint": [
        comparison.loc["Complaint Full", "three_plus"]
    ]
})

chart2c

,Type,Routine,Complaint
0,Share of inspections,13.2,18.84


In [40]:
chart2c.to_csv(
    "../data/processed/chart2c_three_plus.csv",
    index=False
)

In [41]:
chart3 = pd.DataFrame({
    "Result": [
        "No high-priority violations",
        "One or more high-priority violations"
    ],
    "Percent": [
        (complaints["HIGH_VIOL"] == 0).mean() * 100,
        (complaints["HIGH_VIOL"] >= 1).mean() * 100
    ]
})

chart3

,Result,Percent
0,No high-priority violations,47.798226
1,One or more high-priority violations,52.126014


In [42]:
chart3.to_csv(
    "../data/processed/chart3_high_priority_results.csv",
    index=False
)

In [43]:
chart4 = (
    complaints["DISPOSITION"]
    .value_counts()
    .rename_axis("Disposition")
    .reset_index(name="Inspections")
)

chart4["Percent"] = (
    chart4["Inspections"]
    / chart4["Inspections"].sum()
    * 100
).round(1)

chart4

,Disposition,Inspections,Percent
0,Inspection Completed - No Further Action,29058,45.9
1,Call Back - Complied,10978,17.3
2,Warning Issued,9062,14.3
3,Administrative complaint recommended,5022,7.9
4,Call Back - Admin. complaint recommended,2289,3.6
5,Emergency order recommended,2044,3.2
6,"Call Back - Extension given, pending",2008,3.2
7,Emergency Order Callback Complied,1527,2.4
8,Emergency Order Callback Not Complied,960,1.5
9,Emergency Order Callback Time Extension,311,0.5


In [44]:
chart4 = complaints.copy()

def simplify_disposition(x):
    x = str(x)

    if "No Further Action" in x:
        return "No further action"

    elif "Warning" in x:
        return "Warning issued"

    elif "Emergency order recommended" in x:
        return "Emergency order"

    elif "Admin" in x:
        return "Administrative complaint"

    elif "Call Back" in x or "Callback" in x or "Assigned to Inspector" in x:
        return "Follow-up required"

    else:
        return "Other"

chart4["Outcome"] = chart4["DISPOSITION"].apply(simplify_disposition)

chart4 = (
    chart4.groupby("Outcome", as_index=False)
    .size()
    .rename(columns={"size": "Inspections"})
)

chart4["Percent"] = (
    chart4["Inspections"]
    / chart4["Inspections"].sum()
    * 100
).round(1)

chart4 = chart4.sort_values("Percent", ascending=False)

chart4

,Outcome,Inspections,Percent
3,No further action,29058,45.9
2,Follow-up required,15838,25.0
5,Warning issued,9067,14.3
0,Administrative complaint,7336,11.6
1,Emergency order,2044,3.2
4,Other,15,0.0


In [45]:
chart4["Outcome"] = chart4["Outcome"].replace({
    "Follow-up required": "Follow-up inspection",
    "Emergency order": "Emergency closure recommended"
})

In [46]:
chart4.to_csv(
    "../data/processed/chart4_complaint_outcomes.csv",
    index=False
)